# ADM and BSSN Conversions

BSSN rewrites ADM data into variables better suited for stable numerical evolution: a conformal metric perturbation, a conformal factor, the trace-free part of the extrinsic curvature, the trace of the extrinsic curvature, and rescaled shift variables. NRPy exposes these maps through `ADM_to_BSSN` and `BSSN_to_ADM`.

## Table of Contents

1. [Configure the conformal factor](#Configure-the-conformal-factor)
2. [Convert Brill-Lindquist ADM data](#Convert-Brill-Lindquist-ADM-data)
3. [Inspect the generic inverse map](#Inspect-the-generic-inverse-map)
4. [Bounded toy inverse check](#Bounded-toy-inverse-check)

## Configure the Conformal Factor

`ADM_to_BSSN` registers the `EvolvedConformalFactor_cf` parameter through the BSSN quantities module. Import it before setting the parameter so the notebook starts from a clean kernel state.

In [ ]:
import sympy as sp

import nrpy.indexedexp as ixp
import nrpy.params as par
from nrpy.equations.general_relativity.ADM_to_BSSN import ADM_to_BSSN
from nrpy.equations.general_relativity.BSSN_to_ADM import BSSN_to_ADM
from nrpy.equations.general_relativity.InitialData_Cartesian import (
    InitialData_Cartesian,
)

par.set_parval_from_str("EvolvedConformalFactor_cf", "W")
print("EvolvedConformalFactor_cf:", par.parval_from_str("EvolvedConformalFactor_cf"))

## Convert Brill-Lindquist ADM Data

Brill-Lindquist data give a physical, conformally flat ADM example. The conversion prints representative BSSN quantities without expanding every tensor component.

In [ ]:
brill_lindquist = InitialData_Cartesian("BrillLindquist")
adm_to_bssn = ADM_to_BSSN(
    brill_lindquist.gammaDD,
    brill_lindquist.KDD,
    brill_lindquist.betaU,
    brill_lindquist.BU,
    "Cartesian",
)

print("cf:", adm_to_bssn.cf)
print("hDD[0][0]:", adm_to_bssn.hDD[0][0])
print("hDD[0][1]:", adm_to_bssn.hDD[0][1])
print("aDD[0][0]:", adm_to_bssn.aDD[0][0])
print("trK:", adm_to_bssn.trK)
print("vetU:", adm_to_bssn.vetU)
print("betU:", adm_to_bssn.betU)

## Inspect the Generic Inverse Map

`BSSN_to_ADM("Cartesian")` constructs ADM expressions from generic BSSN symbols. These expressions are independent of a particular initial-data choice until substitutions are applied.

In [ ]:
bssn_to_adm = BSSN_to_ADM("Cartesian")

print("gammaDD[0][0]:", bssn_to_adm.gammaDD[0][0])
print("KDD[0][0]:", bssn_to_adm.KDD[0][0])
print("betaU[0]:", bssn_to_adm.betaU[0])

## Bounded Toy Inverse Check

A conformally flat toy metric keeps the inverse-map check small. The toy ADM data use `gammaDD = psi**4 * diag(1, 1, 1)`, zero extrinsic curvature, zero shift, and zero auxiliary shift. The generic inverse map is then evaluated with all `hDD*` and `aDD*` symbols set to zero, `trK` set to zero, and `cf` set to the conformal factor produced by `ADM_to_BSSN`.

In [ ]:
psi = sp.symbols("psi", positive=True)
toy_gammaDD = ixp.zerorank2()
toy_KDD = ixp.zerorank2()
toy_betaU = ixp.zerorank1()
toy_BU = ixp.zerorank1()
for i in range(3):
    toy_gammaDD[i][i] = psi**4

toy_adm_to_bssn = ADM_to_BSSN(
    toy_gammaDD,
    toy_KDD,
    toy_betaU,
    toy_BU,
    "Cartesian",
)
toy_bssn_to_adm = BSSN_to_ADM("Cartesian")

substitutions = {}
generic_expressions = []
for i in range(3):
    generic_expressions.extend(toy_bssn_to_adm.gammaDD[i])
    generic_expressions.extend(toy_bssn_to_adm.KDD[i])
generic_expressions.extend(toy_bssn_to_adm.betaU)

for expression in generic_expressions:
    for symbol in expression.free_symbols:
        name = symbol.name
        if name.startswith("hDD") or name.startswith("aDD"):
            substitutions[symbol] = 0
        elif name == "trK":
            substitutions[symbol] = 0
        elif name == "cf":
            substitutions[symbol] = toy_adm_to_bssn.cf
        elif name.startswith("vetU") or name.startswith("betU"):
            substitutions[symbol] = 0

metric_residuals = [
    sp.factor(toy_bssn_to_adm.gammaDD[i][i].xreplace(substitutions) - toy_gammaDD[i][i])
    for i in range(3)
]
curvature_residuals = [
    sp.factor(toy_bssn_to_adm.KDD[i][i].xreplace(substitutions) - toy_KDD[i][i])
    for i in range(3)
]
shift_residuals = [
    sp.factor(toy_bssn_to_adm.betaU[i].xreplace(substitutions) - toy_betaU[i])
    for i in range(3)
]

print("toy cf:", toy_adm_to_bssn.cf)
print("toy hDD[0][0]:", toy_adm_to_bssn.hDD[0][0])
print("toy aDD[0][0]:", toy_adm_to_bssn.aDD[0][0])
print("toy trK:", toy_adm_to_bssn.trK)
print("metric diagonal residuals:", metric_residuals)
print("KDD diagonal residuals:", curvature_residuals)
print("shift residuals:", shift_residuals)